# agentic_alpha_signed_session_search_v1

Diagnostic R-G-E-V-D card for signed session momentum and Tug-of-War variants. The card retrieves SOXL/SOXX OHLC/VIX state through qlib `D.features`, evaluates long/short overnight, intraday, and close-to-close session exposures from `src.factor.session_effect`, and validates each branch against `qlib_paper_sim_v1` with the MaxDD >= -20% gate.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.factor import (
    build_session_effect_frame,
    compact_session_effect_summary,
    evaluate_signed_session_effect_candidates,
)
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_signed_session_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
START = '2019-01-02'
END = '2026-06-09'
SOURCE_START = '2010-03-11'
SOURCE_END = '2026-06-10'
MAX_DRAWDOWN_FLOOR = -20.0

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  test:
    start: {START}
    end: {END}
rgevd:
  branch: signed_session_momentum
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Route external ideas through retrieve, generate, evaluate, validate, and distill before promotion.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'github',
            'title': 'TSMOM reference implementation',
            'url': 'https://github.com/rkohli3/TSMOM',
            'extracted_signal': 'Use time-series momentum as signed exposure rather than long-only trend gates.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Time-series momentum evidence',
            'url': 'https://papers.ssrn.com/sol3/Delivery.cfm/2730304.pdf?abstractid=2730304',
            'extracted_signal': 'Past returns can define positive or negative exposure by asset and horizon.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Tug of War: overnight versus intraday returns',
            'url': 'https://personal.lse.ac.uk/polk/research/TugOfWar.pdf',
            'extracted_signal': 'Separate overnight and intraday return pressure can carry different information.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Overnight/intraday return reversal trading summary',
            'url': 'https://www.cxoadvisory.com/technical-trading/overnightintraday-return-reversal-trading/',
            'extracted_signal': 'Session returns may reverse across overnight and intraday intervals.',
            'retrieval_status': 'web_retrieved',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
qlib_frame = D.features(
    ['SOXL', 'SOXX'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time=SOURCE_START,
    end_time=SOURCE_END,
    freq='day',
)
features = build_session_effect_frame(qlib_frame, start=START, end=END)
metrics = evaluate_signed_session_effect_candidates(
    features,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    momentum_windows=[5, 10, 20, 40, 60],
    tug_windows=[5, 20, 40, 60],
    thresholds=[0.0, 0.002, 0.005],
    neutral_bands=[0.0, 0.005, 0.01, 0.02],
    trend_filters=['none', 'soxx20', 'soxx60', 'soxl20', 'soxl60'],
    vix_maxes=[22.0, 26.0, 30.0, 35.0, 45.0, 100.0],
    scales=[0.25, 0.5, 1.0, 1.5, 2.5, 3.0],
)
branches = sorted(metrics['branch'].unique())
search_summary = compact_session_effect_summary(
    metrics,
    branches=branches,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
search_summary.update(
    {
        'workflow': 'agentic_alpha_signed_session_search',
        'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
        'feature_evidence': {
            'feature_rows': int(len(features)),
            'metric_rows': int(len(metrics)),
            'source_coverage_start': features.attrs.get('source_coverage_start'),
            'source_coverage_end': features.attrs.get('source_coverage_end'),
            'test_start': START,
            'test_end': END,
            'sessions': ['close_to_next_open', 'open_to_close', 'close_to_next_close'],
        },
        'baseline': {
            'name': baseline.name,
            'return_pct': float(baseline.return_pct),
            'max_drawdown_pct': float(baseline.max_drawdown_pct),
            'benchmark_return_pct': float(baseline.benchmark_return_pct),
            'recorder_id': baseline.recorder_id,
        },
        'passes_both_count': int(len(metrics.loc[metrics['beats_baseline_return'] & metrics['passes_drawdown_gate']])),
    }
)
def compact_candidate_metrics(full_grid: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for branch in sorted(full_grid['branch'].unique()):
        branch_rows = full_grid.loc[full_grid['branch'] == branch].copy()
        best_return = branch_rows.sort_values('return_pct', ascending=False).head(1).assign(selection_reason='best_return')
        drawdown_rows = branch_rows.loc[branch_rows['passes_drawdown_gate'] == True].sort_values(  # noqa: E712
            'return_pct',
            ascending=False,
        )
        if not drawdown_rows.empty:
            pieces.append(drawdown_rows.head(1).assign(selection_reason='best_drawdown_gate'))
        pieces.append(best_return)
    pieces.append(full_grid.sort_values('return_pct', ascending=False).head(20).assign(selection_reason='top_return_grid'))
    pieces.append(
        full_grid.loc[full_grid['passes_drawdown_gate'] == True]  # noqa: E712
        .sort_values('return_pct', ascending=False)
        .head(20)
        .assign(selection_reason='top_drawdown_gate_grid')
    )
    compact = pd.concat(pieces, ignore_index=True, sort=False)
    return compact.drop_duplicates(subset=['candidate']).sort_values(
        ['selection_reason', 'return_pct'],
        ascending=[True, False],
    )

compact_metrics = compact_candidate_metrics(metrics)
full_grid_path = CARD / 'agentic_alpha_signed_session_full_candidate_grid.csv.gz'
metrics.to_csv(full_grid_path, index=False, compression='gzip')
compact_metrics.to_csv(CARD / 'agentic_alpha_signed_session_candidate_metrics.csv', index=False)
search_summary['root_candidate_rows'] = int(len(compact_metrics))
search_summary['full_grid_rows'] = int(len(metrics))
(CARD / 'agentic_alpha_signed_session_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)

failure_modes = (
    '# Failure Modes\n\n'
    '- signed_overnight_momentum: high-return variants still accept deep SOXL path drawdowns; 20% DD-gated variants retain too little exposure to beat the production baseline.\n'
    '- signed_intraday_momentum: intraday signed exposure is lower-return after turnover and does not close the return gate.\n'
    '- signed_tug_of_war_close_to_close: Tug-of-War close-to-close variants improve selectivity but remain far below the production return target inside the drawdown gate.\n'
    '- next_action: do not promote signed session effects alone; combine only if a new predictive model improves crash avoidance without relying on extreme leverage.\n'
)
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        source_url='https://github.com/JacobDu/agentic-alpha',
        metric_rows=len(metrics),
    )
    R.log_metrics(
        passes_both_count=float(search_summary['passes_both_count']),
        best_signed_overnight_return_pct=float(search_summary['branch_results']['signed_overnight_momentum']['best_return_pct']),
        best_signed_overnight_dd_gate_return_pct=float(search_summary['branch_results']['signed_overnight_momentum']['best_drawdown_gate_return_pct']),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_signed_session_search_summary.json').write_text(
        json.dumps(search_summary, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    for path in [
        CARD / 'agentic_alpha_signed_session_candidate_metrics.csv',
        CARD / 'agentic_alpha_signed_session_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_signed_session')
    recorder.log_artifact(full_grid_path.as_posix(), artifact_path='agentic_alpha_signed_session')
    recorder_local_dir = Path(recorder.get_local_dir())

if full_grid_path.exists():
    full_grid_path.unlink()

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(
    path.relative_to(recorder_local_dir).as_posix()
    for path in recorder_local_dir.rglob('*')
    if path.is_file()
)
search_summary['recorder_artifacts'] = recorder_artifacts
search_summary['recorder_id'] = recorder_id
search_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
(CARD / 'agentic_alpha_signed_session_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)

passes_both_count = search_summary['passes_both_count']
promotion_decision = 'promoted' if passes_both_count > 0 else 'not_promoted'
best_overnight = search_summary['branch_results']['signed_overnight_momentum']
best_intraday = search_summary['branch_results']['signed_intraday_momentum']
best_tug = search_summary['branch_results']['signed_tug_of_war_close_to_close']
conclusion = f'''# 中文结论

- 做了什么：继续 R-G-E-V-D，把 time-series momentum、overnight/intraday reversal 和 Tug-of-War 的外部知识转成 signed session long/short 候选，并通过 qlib `D.features` 读取 SOXL/SOXX OHLC/VIX 后调用 `src.factor.evaluate_signed_session_effect_candidates`。
- 核心指标：生产基准 {baseline.name} full-test return_pct={baseline.return_pct:.4f}、max_drawdown_pct={baseline.max_drawdown_pct:.4f}；本卡 passes_both_count={passes_both_count}。
- 分支结果：signed_overnight_momentum 最高收益 return_pct={best_overnight['best_return_pct']:.4f}、max_drawdown_pct={best_overnight['best_return_max_drawdown_pct']:.4f}，最佳 20% 回撤候选 return_pct={best_overnight['best_drawdown_gate_return_pct']:.4f}、max_drawdown_pct={best_overnight['best_drawdown_gate_max_drawdown_pct']:.4f}；signed_intraday_momentum 最佳 20% 回撤候选 return_pct={best_intraday['best_drawdown_gate_return_pct']:.4f}；signed_tug_of_war_close_to_close 最佳 20% 回撤候选 return_pct={best_tug['best_drawdown_gate_return_pct']:.4f}。
- regime/window 差异：overnight signed momentum 是唯一有明显收益弹性的 session 分支，但高收益路径仍在 2020/2022/2025-2026 压力段承受深回撤；20% DD 内的 0.5x signed overnight 只能保留约数百个百分点收益，无法超过生产 baseline。
- 是否晋升：{promotion_decision}；未找到同时满足收益超过现有生产策略且 MaxDD >= -20% 的候选。
- 风险：这是 diagnostic search，不是生产门；signed session 单独使用无法达成目标，后续只能作为新模型的候选特征或风险解释，不应直接提升为 production strategy。
- qlib recorder：{recorder_id}
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'market': 'all',
    'region': 'us',
    'as_of': END,
    'data_range': [START, END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'workflow_config': 'workflow.yaml',
    'baseline': search_summary['baseline'],
    'search_report': search_summary,
    'branch_results': search_summary['branch_results'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'promotion_decision': promotion_decision,
    'external_source_counts': search_summary['external_source_counts'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'recorder': 'https://qlib.readthedocs.io/en/latest/component/recorder.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_signed_session_candidate_metrics.csv',
            'agentic_alpha_signed_session_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
        ],
        'recorder_only_artifacts': {
            'full_candidate_grid': 'artifacts/agentic_alpha_signed_session/agentic_alpha_signed_session_full_candidate_grid.csv.gz',
            'predictions': 'artifacts/pred.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_signed_session_candidate_metrics.csv',
        'search_summary': 'agentic_alpha_signed_session_search_summary.json',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
print(conclusion)
print('candidate_metrics=' + (CARD / 'agentic_alpha_signed_session_candidate_metrics.csv').name)
print('summary=summary.json')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[64152:MainThread](2026-06-23 19:46:47,955) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[64152:MainThread](2026-06-23 19:46:47,956) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[64152:MainThread](2026-06-23 19:46:47,956) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[64152:MainThread](2026-06-23 19:46:53,638) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_signed_session_search_v1.


[64152:MainThread](2026-06-23 19:46:53,640) INFO - qlib.workflow - [exp.py:258] - Experiment 684423351917192036 starts running ...


[64152:MainThread](2026-06-23 19:46:53,711) INFO - qlib.workflow - [recorder.py:345] - Recorder ec61dbaee29d4737861261e112a98314 starts running under Experiment 684423351917192036 ...


[64152:MainThread](2026-06-23 19:46:53,786) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


# 中文结论

- 做了什么：继续 R-G-E-V-D，把 time-series momentum、overnight/intraday reversal 和 Tug-of-War 的外部知识转成 signed session long/short 候选，并通过 qlib `D.features` 读取 SOXL/SOXX OHLC/VIX 后调用 `src.factor.evaluate_signed_session_effect_candidates`。
- 核心指标：生产基准 qlib_paper_sim_v1 full-test return_pct=6570.0233、max_drawdown_pct=-85.2317；本卡 passes_both_count=0。
- 分支结果：signed_overnight_momentum 最高收益 return_pct=230037.2597、max_drawdown_pct=-94.4088，最佳 20% 回撤候选 return_pct=451.1520、max_drawdown_pct=-15.6807；signed_intraday_momentum 最佳 20% 回撤候选 return_pct=40.9092；signed_tug_of_war_close_to_close 最佳 20% 回撤候选 return_pct=129.2798。
- regime/window 差异：overnight signed momentum 是唯一有明显收益弹性的 session 分支，但高收益路径仍在 2020/2022/2025-2026 压力段承受深回撤；20% DD 内的 0.5x signed overnight 只能保留约数百个百分点收益，无法超过生产 baseline。
- 是否晋升：not_promoted；未找到同时满足收益超过现有生产策略且 MaxDD >= -20% 的候选。
- 风险：这是 diagnostic search，不是生产门；signed session 单独使用无法达成目标，后续只能作为新模型的候选特征或风险解释，不应直接提升为 production strategy。
- qlib recorder：ec61dbaee29d4737861261e112a98314

candi

## Final Chinese Conclusion

Executed output above is the durable conclusion. Keep large predictions, signals, and portfolio reports in qlib recorder only; this card keeps slim sidecars plus `summary.json`.